# Tutorial 2: Composition

This tutorial builds on [Tutorial 1 (counter)](counter.ipynb). We define a pendulum environment and a controller neural network, then **compose** them into a closed-loop system using shared wires.

You'll learn:
1. How modules get their own fresh wires by default
2. How to create shared wire pairs and pass them to both modules
3. How `Module.parallel` merges modules that share wires

## The pendulum environment

A simplified inverted pendulum using the small-angle approximation ($\sin\theta \approx \theta$):

$$\dot\theta' = \dot\theta + \left(-\frac{g}{\ell}\,\theta + \frac{\tau}{m\ell^2}\right) \Delta t, \qquad \theta' = \theta + \dot\theta' \, \Delta t$$

Same `Env` contract as the counter: `__init__` sets spaces and state, `reset` initializes, `step` updates.

In [ ]:
import torch
import torch.nn as nn
from gymnasium import spaces
from zrth import Env, NN, Module


class PendulumEnv(Env):
    """Simplified inverted pendulum (small-angle approximation)."""

    def __init__(self, g=9.81, l=1.0, m=1.0, dt=0.05):
        super().__init__()

        self.action_space = spaces.Box(low=-2.0, high=2.0, shape=(1,))
        self.observation_space = spaces.Box(low=-10.0, high=10.0, shape=(1,))

        self.g = g
        self.l = l
        self.m = m
        self.dt = dt

        # Also set state variables here so the analyzer can infer their dtype
        self.theta = 0.1
        self.theta_dot = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.theta = 0.1
        self.theta_dot = 0.0

        observation = self.theta
        reward = 0.0
        terminated = False
        truncated = False
        return observation, reward, terminated, truncated

    def step(self, torque):
        # Small-angle approximation: sin(theta) ~ theta
        accel = (0.0 - self.g / self.l) * self.theta + torque / (self.m * self.l * self.l)
        self.theta_dot = self.theta_dot + accel * self.dt
        self.theta = self.theta + self.theta_dot * self.dt

        observation = self.theta
        reward = 0.0 - self.theta * self.theta - 0.1 * self.theta_dot * self.theta_dot
        terminated = self.theta > 3.14 or self.theta < -3.14
        truncated = False
        return observation, reward, terminated, truncated

Instantiate and inspect — same as in Tutorial 1.

In [ ]:
pendulum = PendulumEnv(g=9.81, l=1.0, m=1.0, dt=0.05)
print(pendulum)

## The controller

A small NN that observes $\theta$ and outputs a torque $\tau$. Architecture: $[1] \to 4 \to [1]$.

In [ ]:
class ControllerNN(NN):
    """NN controller: theta -> torque."""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 4)
        self.fc2 = nn.Linear(4, 1)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        return self.fc2(x)

In [ ]:
controller = ControllerNN()
print(Module.__str__(controller))

## Composition with shared wires

By default, each `Env` and `NN` creates its own fresh wires. To connect them into a closed loop, we create wire pairs **up front** and pass them to both constructors:

- `Env` accepts `observation=` and `action=` kwargs
- `NN` accepts `extl=` and `intf=` kwargs

A wire pair is `[Wire(dtype), Wire(dtype)]` — one latched, one next. By passing the **same** pair as the pendulum's `observation` and the controller's `extl`, the env's observation output feeds the controller's input. Similarly, the controller's `intf` output feeds the env's `action` input.

`Module.parallel` then merges both modules. Because they share wires, the result is a single closed-loop system.

In [ ]:
from zrth import Wire, Float

# Shared wire pairs: pendulum observation = controller input
obs_wire = [Wire(Float(1)), Wire(Float(1))]
# Shared wire pairs: controller output = pendulum action
act_wire = [Wire(Float(1)), Wire(Float(1))]

pendulum = PendulumEnv(g=9.81, l=1.0, m=1.0, dt=0.05, observation=obs_wire, action=act_wire)
controller = ControllerNN(extl=obs_wire, intf=act_wire)

closed_loop = Module.parallel(pendulum, controller)
print(closed_loop)

## Training

The loop will:

1. Simulate the closed-loop system for $N$ steps
2. Collect $(\theta, \dot\theta, \tau, r)$ trajectories
3. Use policy gradient (or similar) to train the controller NN
4. Update controller weights and simulate again to verify

In [ ]:
# TODO: training loop
# for epoch in range(n_epochs):
#     trajectory = simulate(closed_loop, n_steps)
#     for theta, theta_dot, torque, reward in trajectory:
#         # policy gradient update
#         pass
#     optimizer.step()